In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Install and Import Libraries**

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import wandb

print("Libraries loaded.")

Libraries loaded.


# **Configuration**

In [3]:
DATA_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTION_COLS = ["A", "B", "C", "D", "E"]

# **W&B Setup**

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb.init(
    project="smart-mcq-solver",
    name="model1-tfidf-logistic",
    config={
        "model"        : "LogisticRegression",
        "max_features" : 100000,
        "ngram_range"  : "(1,3)",
        "C"            : 10.0,
        "max_iter"     : 2000,
        "sublinear_tf" : True
    }
)

print("W&B ready.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260628_192618-bmwg1fyv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-tfidf-logistic
wandb: ⭐️ View project at https://wandb.ai/priyamtiwari948-indian-institute-of-technology-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/priyamtiwari948-indian-institute-of-technology-madras/smart-mcq-solver/runs/bmwg1fyv


W&B ready.


# **Load Dataset**

In [5]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train:", train.shape)
print("Test :", test.shape)
train.head(3)

Train: (2000, 8)
Test : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


# **Text Preprocessing**

In [6]:
def combine_text(row):
    parts = [str(row['prompt'])] + [str(row[c]) for c in OPTION_COLS]
    return ' '.join(parts)

train['combined'] = train.apply(combine_text, axis=1)
test['combined']  = test.apply(combine_text, axis=1)

print("Text combined.")

Text combined.


#  **TF-IDF Vectorization**

In [7]:
vectorizer = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 3),
    stop_words='english',
    sublinear_tf=True,
    min_df=1,
    analyzer='word'
)

X_train = vectorizer.fit_transform(train['combined'])
X_test  = vectorizer.transform(test['combined'])

le = LabelEncoder()
y_train = le.fit_transform(train['answer'])

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Classes:", le.classes_)

X_train: (2000, 25642)
X_test : (500, 25642)
Classes: ['A' 'B' 'C' 'D' 'E']


# **Train Logistic Regression**

In [8]:
clf = LogisticRegression(
    max_iter=2000,
    C=10.0,
    solver='lbfgs',
    multi_class='multinomial',
    n_jobs=-1
)

clf.fit(X_train, y_train)
print("Model trained.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Model trained.


 #  **Evaluate MAP@3** 

In [9]:
def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

probs      = clf.predict_proba(X_train)
top3_idx   = np.argsort(probs, axis=1)[:, ::-1][:, :3]
top3_preds = [[le.classes_[i] for i in row] for row in top3_idx]

scores     = [map_at_3(true, pred) for true, pred in zip(train['answer'], top3_preds)]
train_map3 = np.mean(scores)
print(f"Train MAP@3: {train_map3:.4f}")

wandb.log({"train_map3": train_map3})

Train MAP@3: 1.0000


# **Submission**

In [10]:
probs_test = clf.predict_proba(X_test)
top3_idx   = np.argsort(probs_test, axis=1)[:, ::-1][:, :3]
test_preds = [' '.join([le.classes_[i] for i in row]) for row in top3_idx]

submission = pd.DataFrame({
    'ID'         : test['id'],
    'Prediction' : test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission created.")
print(submission.head(10))

wandb.finish()

wandb: updating run metadata


Submission created.
   ID Prediction
0   1      A B E
1   2      B C A
2   3      B C D
3   4      E C B
4   5      C D E
5   6      D C A
6   7      E C B
7   8      B C D
8   9      C B D
9  10      B C D


wandb: uploading history steps 0-0, summary, console lines 1-22
wandb: 
wandb: Run history:
wandb: train_map3 ▁
wandb: 
wandb: Run summary:
wandb: train_map3 1
wandb: 
wandb: 🚀 View run model1-tfidf-logistic at: https://wandb.ai/priyamtiwari948-indian-institute-of-technology-madras/smart-mcq-solver/runs/bmwg1fyv
wandb: ⭐️ View project at: https://wandb.ai/priyamtiwari948-indian-institute-of-technology-madras/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260628_192618-bmwg1fyv/logs
